# Transfer Learning with MobileNetV2 (Keras/TensorFlow)
This notebook trains a **MobileNetV2**-based classifier on CIFAR‑10 (resized to 224×224) as a reproducible demo.
It includes augmentation, training curves, a confusion matrix, and an inference example on a sample image.

Replace CIFAR‑10 with your dataset by following the **Custom Dataset** cell.

In [ ]:

import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print(tf.__version__)
seed = 42
tf.keras.utils.set_random_seed(seed)
IMG_SIZE = 224
BATCH = 32
NUM_CLASSES = 10  # CIFAR-10


In [ ]:

# --- Data: CIFAR-10 (resized to 224x224) ---
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze(); y_test = y_test.squeeze()

# small validation split
val_frac = 0.2
n_val = int(len(x_train) * val_frac)
x_val, y_val = x_train[:n_val], y_train[:n_val]
x_train, y_train = x_train[n_val:], y_train[n_val:]

def resize(imgs):
    imgs = tf.image.resize(imgs, (IMG_SIZE, IMG_SIZE)).numpy().astype("uint8")
    return imgs

x_train_r = resize(x_train)
x_val_r   = resize(x_val)
x_test_r  = resize(x_test)

print(x_train_r.shape, x_val_r.shape, x_test_r.shape)


In [ ]:

# --- Data pipeline with augmentation ---
data_augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augment")

def make_ds(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(4096, seed=42)
    ds = ds.map(lambda a,b: (preprocess_input(tf.cast(a, tf.float32)), b), num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda a,b: (data_augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(x_train_r, y_train, training=True)
val_ds   = make_ds(x_val_r,   y_val,   training=False)
test_ds  = make_ds(x_test_r,  y_test,  training=False)


In [ ]:

# --- Model: MobileNetV2 backbone (frozen) + small head ---
base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


In [ ]:

# --- Train ---
ckpt = keras.callbacks.ModelCheckpoint("checkpoints/best.keras", save_best_only=True, monitor="val_accuracy", mode="max")
es   = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy", mode="max")
hist = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[ckpt, es])


In [ ]:

# --- Curves ---
plt.figure(figsize=(6,4))
plt.plot(hist.history['accuracy'], label='train acc')
plt.plot(hist.history['val_accuracy'], label='val acc')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.title('Accuracy'); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist.history['loss'], label='train loss')
plt.plot(hist.history['val_loss'], label='val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('Loss'); plt.legend(); plt.show()


In [ ]:

# --- Evaluate on test ---
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print("Test accuracy:", round(test_acc, 3))

# Confusion matrix (on a small subset for speed)
x_s, y_s = x_test_r[:200], y_test[:200]
preds = model.predict(preprocess_input(x_s.astype("float32")), verbose=0).argmax(axis=1)
cm = confusion_matrix(y_s, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(xticks_rotation=45)
plt.title("Confusion Matrix (subset)")
plt.show()

print(classification_report(y_s, preds))


In [ ]:

# --- Inference demo on a sample image (assets/sample.jpg) ---
from PIL import Image
img = Image.open("../assets/sample.jpg").convert("RGB").resize((224,224))
arr = np.array(img).astype("float32")
arr = preprocess_input(arr[None, ...])
probs = model.predict(arr, verbose=0)[0]
pred = probs.argmax()
print("Predicted class index:", int(pred), " | confidence:", float(probs[pred]))


## Custom Dataset (optional)
To use your own dataset, structure folders like:

```
data/
  train/
    class_a/ ...images...
    class_b/ ...images...
  val/
    class_a/ ...images...
    class_b/ ...images...
```

Then replace the CIFAR‑10 section with `image_dataset_from_directory` and keep the rest of the code.
You can also unfreeze the last few layers of MobileNetV2 for fine‑tuning (small learning rate).